# Tạo giọng thuyết minh — *Sparse Feature Circuits* (đọc theo đoạn)

Mỗi cảnh được đọc liền một lần để ngữ điệu tự nhiên, rồi tự cắt thành từng câu theo khoảng lặng.

Bật GPU: **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**. Xong sẽ tự tải về `tts_mp3.zip`.

In [ ]:
!pip -q install zerotts soundfile

In [ ]:
SCENES = [
 [
  "intro",
  [
   "Chào bạn! Hôm nay mình cùng mổ xẻ một paper về AI có thể giải thích được, đăng ở hội nghị ai xi eo a hai nghìn không trăm hai mươi lăm.",
   "Tên paper là Sparse Feature Circuits, tạm dịch: các mạch đặc trưng thưa.",
   "Nghe thì đáng sợ, nhưng mình sẽ giải thích từng khái niệm một, trước khi dùng nó."
  ]
 ],
 [
  "lm",
  [
   "Trước hết, mô hình ngôn ngữ là gì? Đó là một chương trình đoán từ tiếp theo trong câu.",
   "Ví dụ, sau câu: The boys near the teacher..., mô hình phải chọn giữa have và has.",
   "Chủ ngữ là boys, số nhiều, nên đáp án đúng là have. Mô hình thường chọn đúng.",
   "Nhưng câu hỏi là: bên trong nó đã làm gì để ra được quyết định đó?",
   "Bên trong mô hình chỉ là hàng triệu con số chạy qua nhiều tầng. Nhìn vào đó, con người chẳng hiểu gì cả. Đây gọi là vấn đề hộp đen.",
   "Lĩnh vực ích ây ai, hay AI giải thích được, cố gắng mở chiếc hộp đen này ra."
  ]
 ],
 [
  "circuit_def",
  [
   "Một cách mở hộp đen là tìm mạch, hay circuit.",
   "Mạch là một nhóm nhỏ các bộ phận bên trong mô hình, cùng các đường nối giữa chúng, đủ để giải thích một hành vi cụ thể.",
   "Giống như trong một thành phố khổng lồ, ta chỉ cần vẽ đúng những con đường mà chiếc xe đã đi qua."
  ]
 ],
 [
  "problem",
  [
   "Các nghiên cứu trước đây vẽ mạch bằng những bộ phận thô, như neuron hay attention head.",
   "Vấn đề thứ nhất: một neuron thường đa nghĩa. Cùng một neuron có thể sáng lên với con mèo, với xe hơi, lẫn tiếng Pháp.",
   "Vậy nên dù tìm ra mạch, ta vẫn không hiểu mạch đó đang làm gì.",
   "Vấn đề thứ hai: nếu chia nhỏ ra thành đơn vị tinh hơn thì có hàng chục nghìn đơn vị. Thử từng cái một thì tốn thời gian khủng khiếp.",
   "Paper này giải quyết cả hai vấn đề: dùng Sparse Autoencoder để có đơn vị dễ hiểu, và dùng xấp xỉ tuyến tính để tính cho nhanh."
  ]
 ],
 [
  "vector",
  [
   "Trước khi vào Sparse Autoencoder, mình ôn nhanh khái niệm vector.",
   "Vector đơn giản là một danh sách các con số. Với hai con số, ta vẽ nó thành một mũi tên trên mặt phẳng.",
   "Hai vector có thể cộng lại: đặt mũi tên nối đuôi nhau, ta được một mũi tên mới.",
   "Và ta có thể nhân vector với một số, gọi là hệ số, để kéo dài hoặc thu ngắn nó.",
   "Trong mô hình ngôn ngữ, trạng thái bên trong tại mỗi từ là một vector rất dài, gọi là activation, ký hiệu x. Với Pythia, nó có 512 con số."
  ]
 ],
 [
  "sae",
  [
   "Ý tưởng của Sparse Autoencoder, viết tắt là ét a e, như sau.",
   "Ta chuẩn bị sẵn một cuốn từ điển gồm rất nhiều vector hướng, mỗi hướng là một feature, hay đặc trưng, ký hiệu v i.",
   "Rồi ta biểu diễn activation x thành tổng của một vài feature, mỗi cái nhân với một hệ số f i.",
   "Chữ sparse, nghĩa là thưa, tức là hầu hết hệ số bằng 0. Mỗi lần chỉ vài feature được bật lên.",
   "Viết thành công thức: x bằng tổng các f i nhân v i, cộng b, cộng epsilon.",
   "Ký hiệu Sigma nghĩa là cộng dồn qua tất cả các feature. b là độ lệch cố định. Còn epsilon là phần sai số mà từ điển chưa diễn tả được.",
   "Điều kỳ diệu là các feature này thường dễ hiểu với con người, ví dụ: feature bật lên khi gặp danh từ số nhiều."
  ]
 ],
 [
  "sae_train",
  [
   "ét a e học cuốn từ điển đó bằng cách tối ưu hai mục tiêu cùng lúc.",
   "Mục tiêu một: bản tái tạo x mũ phải gần với x thật, tức là sai số bình phương nhỏ.",
   "Mục tiêu hai: phạt việc dùng quá nhiều feature, để các hệ số f i luôn thưa.",
   "Từ điển rất rộng: với Pythia, số feature gấp 64 lần số chiều. Với Gemma 2, tác giả dùng bộ ét a e có sẵn tên Gemma Scope."
  ]
 ],
 [
  "graph",
  [
   "Và đây là mẹo quan trọng đầu tiên của paper.",
   "Ta chèn ét a e vào nhiều vị trí trong mô hình: attention, em eo pi và dòng residual ở mỗi tầng.",
   "Khi đó, mỗi feature, và cả phần sai số epsilon, trở thành một nút trong đồ thị tính toán. Hình vuông là feature, hình tam giác là sai số.",
   "Việc tìm mạch bây giờ chỉ là: chọn ra những nút và cạnh quan trọng nhất trong đồ thị này."
  ]
 ],
 [
  "metric",
  [
   "Nhưng quan trọng với cái gì? Ta cần một thước đo, ký hiệu m.",
   "Mô hình gán cho mỗi từ một xác suất. Ta lấy log xác suất của have trừ đi log xác suất của has.",
   "Nếu m dương và lớn, mô hình nghiêng mạnh về have. Nếu âm, nó nghiêng về has.",
   "Log chỉ là một cách co giãn thang đo cho dễ so sánh, bạn không cần lo về nó."
  ]
 ],
 [
  "ie",
  [
   "Để đo tầm quan trọng của một nút a, paper dùng khái niệm hiệu ứng gián tiếp, gọi tắt là ai i.",
   "Ta có hai câu gần giống nhau: câu sạch, The teacher, và câu vá, The teachers. Chỉ khác số ít, số nhiều.",
   "Chạy mô hình trên câu sạch. Rồi, đúng tại nút a, ta thay giá trị của nó bằng giá trị lấy từ câu vá. Ký hiệu do nghĩa là: ép nút này nhận giá trị đó.",
   "ai i bằng thước đo m sau khi can thiệp, trừ đi m ban đầu. Nếu m thay đổi nhiều, nút a thật sự có ảnh hưởng.",
   "Đây là suy luận nhân quả: ta không chỉ nhìn tương quan, mà thật sự can thiệp và xem kết quả thay đổi."
  ]
 ],
 [
  "cost",
  [
   "Vấn đề là: làm như vậy thì mỗi nút cần chạy lại cả mô hình một lần.",
   "Với hàng chục nghìn feature ở nhiều vị trí, việc này quá chậm."
  ]
 ],
 [
  "atp",
  [
   "Giải pháp là xấp xỉ tuyến tính, dùng gradient. Gradient nghĩa là độ dốc: nếu nhích a một chút, m thay đổi bao nhiêu.",
   "Hãy tưởng tượng m là một đường cong theo a. Tại điểm sạch, ta vẽ đường tiếp tuyến.",
   "Thay vì thật sự đi tới điểm vá, ta đi dọc theo tiếp tuyến: độ dốc nhân với khoảng cách, là ra ước lượng.",
   "Đó là công thức attribution patching: ai i xấp xỉ bằng gradient của m theo a, nhân với a vá trừ a sạch.",
   "Cái hay: chỉ cần hai lượt chạy xuôi và một lượt lan truyền ngược, ta có ước lượng cho tất cả các nút cùng lúc.",
   "Lan truyền ngược, hay backpropagation, là thuật toán tính gradient cho mọi nút chỉ trong một lần."
  ]
 ],
 [
  "ig",
  [
   "Nhưng nếu đường cong uốn mạnh, tiếp tuyến sẽ đoán sai, nhất là ở các tầng đầu của mô hình.",
   "Khi đó paper dùng integrated gradients, tức là gradient tích phân.",
   "Ta chia đoạn từ a sạch tới a vá thành 10 bước, đo độ dốc tại mỗi bước, rồi lấy trung bình.",
   "Độ dốc trung bình này bám theo đường cong tốt hơn nhiều, đổi lại tốn thêm khoảng 10 lần tính toán."
  ]
 ],
 [
  "filter",
  [
   "Có điểm ai i cho mọi nút rồi, ta lấy trung bình trên nhiều cặp câu.",
   "Sau đó giữ lại những nút có độ lớn ai i vượt ngưỡng tê en. Các nút còn lại bị bỏ đi.",
   "Làm tương tự với các cạnh nối giữa các nút, dùng ngưỡng tê i.",
   "Kết quả là một đồ thị nhỏ gọn: một mạch đặc trưng thưa, gồm các feature dễ hiểu và cách chúng truyền thông tin cho nhau."
  ]
 ],
 [
  "pipeline",
  [
   "Tóm lại, quy trình có bốn bước.",
   "Bước một: chạy mô hình trên câu sạch và câu vá, lưu lại mọi activation.",
   "Bước hai: lan truyền ngược, lưu gradient của m tại mọi nút.",
   "Bước ba: tính ai i xấp xỉ cho từng nút, lọc theo ngưỡng.",
   "Bước bốn: tính ai i cho các cạnh, lọc tiếp. Và ta có mạch."
  ]
 ],
 [
  "eval",
  [
   "Làm sao biết mạch này tốt? Paper dùng hai thước đo.",
   "Thứ nhất là độ trung thành, faithfulness. Ta chỉ giữ lại mạch C, còn mọi nút khác bị thay bằng giá trị trung bình. Gọi là mean ablation.",
   "Faithfulness bằng m của mạch trừ m của mạch rỗng, chia cho m của cả mô hình trừ m của mạch rỗng. Tức là: mạch giải thích được bao nhiêu phần trăm hành vi. Lý tưởng là 1.",
   "Thứ hai là độ đầy đủ, completeness: bỏ mạch đi, xem phần còn lại còn làm được việc không. Lý tưởng là gần 0.",
   "Kết quả: trên Pythia, chỉ khoảng 100 feature đã giải thích phần lớn hành vi, trong khi cần khoảng 1500 neuron mới giải thích được một nửa.",
   "Trên Gemma 2, khoảng 500 feature so với khoảng 50 nghìn neuron. Feature hiệu quả hơn hẳn."
  ]
 ],
 [
  "case",
  [
   "Giờ ta xem một mạch cụ thể: hòa hợp chủ ngữ và động từ qua mệnh đề quan hệ.",
   "Câu mẫu: The girl that the teacher sees, rồi đến has hoặc have. Chú ý: động từ phải hợp với girl, không phải teacher.",
   "Đầu tiên, các feature ở tầng thấp phát hiện số ít hay số nhiều của danh từ girl.",
   "Tiếp theo, các feature nhận ra từ that đang mở đầu một mệnh đề phụ.",
   "Rồi các feature phát hiện mệnh đề phụ đã kết thúc, và mang thông tin số của chủ ngữ tới vị trí cuối.",
   "Cuối cùng, các feature phân biệt dạng động từ đẩy xác suất về has hoặc have.",
   "Mạch này đọc được như một thuật toán mà con người hiểu. Gemma 2 còn dùng thêm các feature theo dõi số của cả cụm danh từ."
  ]
 ],
 [
  "shift_problem",
  [
   "Phần ứng dụng thú vị nhất: Shift, tức là cắt tỉa các feature giả tạo mà con người hiểu được.",
   "Bài toán: phân loại nghề nghiệp từ tiểu sử, giáo sư hay y tá.",
   "Nhưng dữ liệu huấn luyện bị lệch cực độ: toàn giáo sư nam và y tá nữ.",
   "Mô hình sẽ đi đường tắt: đoán theo giới tính thay vì nghề nghiệp. Đó là tín hiệu giả, spurious signal.",
   "Các phương pháp cũ cần thêm dữ liệu có nhãn để gỡ lệch. Ở đây, ta không có dữ liệu đó."
  ]
 ],
 [
  "shift_method",
  [
   "Shift làm như sau. Bước một: tìm mạch đặc trưng giải thích quyết định của bộ phân loại.",
   "Bước hai: con người đọc từng feature trong mạch, và đánh giá nó có liên quan đến nghề nghiệp hay không.",
   "Ví dụ, feature bật lên với từ she, her, hay women là về giới tính, không liên quan tới nghề.",
   "Bước ba: tắt hẳn các feature không liên quan, bằng cách đặt chúng về 0. Bước bốn, tùy chọn: huấn luyện lại lớp phân loại."
  ]
 ],
 [
  "shift_result",
  [
   "Kết quả trên Pythia, đo trên dữ liệu cân bằng.",
   "Độ chính xác đoán nghề tăng từ 62 phần trăm lên 88, và lên 93 sau khi huấn luyện lại, ngang với mô hình lý tưởng.",
   "Khả năng đoán giới tính giảm từ 87 xuống khoảng 52 phần trăm, gần như đoán mò. Nghĩa là thiên kiến giới tính đã gần như biến mất.",
   "Nhóm tệ nhất tăng từ 24 lên 89 phần trăm. Còn nếu tắt feature ngẫu nhiên, hay dùng neuron, thì không đạt được như vậy."
  ]
 ],
 [
  "unsup",
  [
   "Cuối cùng, paper tự động hóa toàn bộ quy trình ở quy mô lớn.",
   "Lấy một kho văn bản khổng lồ, gom các ví dụ có cách dự đoán giống nhau thành từng cụm, không cần con người gán nhãn.",
   "Mỗi cụm là một hành vi, ví dụ: đếm tiếp dãy số một, hai, ba, bốn, hoặc dự đoán từ to sau các động từ.",
   "Rồi tìm mạch cho từng cụm. Kết quả là hàng nghìn mạch cho hàng nghìn hành vi, có thể xem tại trang feature circuits chấm ích i dét."
  ]
 ],
 [
  "limits",
  [
   "Tất nhiên, phương pháp vẫn có hạn chế.",
   "Cần có ét a e được huấn luyện sẵn, tốn nhiều tài nguyên tính toán ban đầu.",
   "Phần sai số epsilon vẫn là hộp đen nhỏ chưa giải thích được.",
   "Và việc gán ý nghĩa cho feature phần lớn vẫn là đánh giá định tính của con người."
  ]
 ],
 [
  "outro",
  [
   "Tóm lại: paper dùng ét a e để có những đơn vị dễ hiểu, dùng gradient để tìm nhanh những đơn vị quan trọng, và nối chúng thành mạch.",
   "Mạch đó giúp ta hiểu mô hình, và còn giúp ta sửa mô hình, như gỡ bỏ thiên kiến giới tính mà không cần thêm dữ liệu.",
   "Cảm ơn bạn đã theo dõi!"
  ]
 ]
]
print(len(SCENES), 'cảnh,', sum(len(l) for _, l in SCENES), 'câu')

In [ ]:
VOICE = 'quangminh'
MAX_CHUNK_SEC = 40        # đoạn dài hơn ~40s sẽ được chia nhỏ
CHARS_PER_SEC = 14        # ước lượng tốc độ đọc để chia đoạn

import os, json, subprocess, shutil, numpy as np, soundfile as sf
from tqdm.auto import tqdm
from IPython.display import Audio, display
from zerotts import ZeroTTS
try:
    from zerotts import normalize_vi_text
except Exception:
    normalize_vi_text = lambda s: s

tts = ZeroTTS.from_pretrained("zeroweight-ai/ZeroTTS")
SR = getattr(tts, 'sample_rate', 48000)

def synth(text):
    a = tts.synthesize(normalize_vi_text(text), voice=VOICE)
    if hasattr(a, 'detach'): a = a.detach().cpu().numpy()
    return np.asarray(a, dtype=np.float32).squeeze()

def frame_db(a, hop):
    n = len(a) // hop
    e = np.sqrt(np.mean(a[:n * hop].reshape(n, hop) ** 2, axis=1) + 1e-10)
    return 20 * np.log10(e / (e.max() + 1e-9))

def trim(a, hop, thr=-45, pad=0.06):
    db = frame_db(a, hop); idx = np.where(db > thr)[0]
    if len(idx) == 0: return a
    s = max(0, idx[0] * hop - int(pad * SR)); e = min(len(a), (idx[-1] + 1) * hop + int(pad * SR))
    return a[s:e]

def pauses(a, hop, thr=-38, min_ms=90):
    """các khoảng lặng bên trong: list (giữa, độ dài giây)"""
    db = frame_db(a, hop); sil = db < thr; out = []; i = 0; n = len(sil)
    while i < n:
        if sil[i]:
            j = i
            while j < n and sil[j]: j += 1
            if i > 0 and j < n and (j - i) * hop / SR * 1000 >= min_ms:
                out.append(((i + j) / 2 * hop / SR, (j - i) * hop / SR))
            i = j
        else: i += 1
    return out

def split_points(a, lines):
    """chọn N-1 ranh giới: khoảng lặng dài + gần vị trí dự kiến theo số chữ"""
    hop = int(SR * 0.01); T = len(a) / SR
    cands = pauses(a, hop)
    L = np.array([len(l) for l in lines], float); exp = np.cumsum(L)[:-1] / L.sum() * T
    pts, used, prev, method = [], set(), 0.0, 'pause'
    for k, e in enumerate(exp):
        best, bs = None, -1e9
        for ci, (c, d) in enumerate(cands):
            if ci in used or c <= prev + 0.3: continue
            s = d * 4 - abs(c - e) / (0.12 * T)
            if s > bs: best, bs = ci, s
        if best is None or abs(cands[best][0] - e) > 0.35 * T:
            pts.append(e); method = 'ratio'
        else:
            used.add(best); pts.append(cands[best][0])
        prev = pts[-1]
    return pts, method

def normalize(a, target_db=-20):
    rms = np.sqrt(np.mean(a ** 2) + 1e-10)
    a = a * (10 ** (target_db / 20) / rms)
    pk = np.abs(a).max()
    return a / pk * 0.95 if pk > 0.95 else a

os.makedirs('tts_mp3', exist_ok=True); os.makedirs('chunks', exist_ok=True)
report = []
for sid, lines in tqdm(SCENES):
    # chia cảnh thành các đoạn <= MAX_CHUNK_SEC
    chunks, cur = [], []
    for i, l in enumerate(lines):
        if cur and sum(len(lines[j]) for j in cur + [i]) / CHARS_PER_SEC > MAX_CHUNK_SEC:
            chunks.append(cur); cur = []
        cur.append(i)
    chunks.append(cur)
    for ci, idxs in enumerate(chunks):
        texts = [lines[i] for i in idxs]
        a = synth(' '.join(texts))
        a = normalize(trim(a, int(SR * 0.01)))
        sf.write(f'chunks/{sid}_c{ci}.wav', a, SR)
        if len(idxs) == 1: pts, method = [], 'single'
        else: pts, method = split_points(a, texts)
        bounds = [0.0] + pts + [len(a) / SR]
        for k, i in enumerate(idxs):
            seg = a[int(bounds[k] * SR):int(bounds[k + 1] * SR)]
            sf.write('tmp.wav', seg, SR)
            subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', 'tmp.wav', '-ac', '1', '-b:a', '160k', f'tts_mp3/{sid}_{i}.mp3'], check=True)
            report.append({'id': f'{sid}_{i}', 'chunk': f'{sid}_c{ci}', 'start': round(bounds[k], 3),
                           'end': round(bounds[k + 1], 3), 'method': method})
json.dump(report, open('tts_mp3/report.json', 'w'), ensure_ascii=False, indent=1)

n = len([f for f in os.listdir('tts_mp3') if f.endswith('.mp3')])
print(n, '/', sum(len(l) for _, l in SCENES), 'file câu')
bad = sorted({r['chunk'] for r in report if r['method'] == 'ratio'})
print('Đoạn phải chia theo tỉ lệ chữ (nên nghe lại):', bad if bad else 'không có')
print('Nghe thử 1 đoạn:'); display(Audio('chunks/sae_c0.wav'))

shutil.make_archive('tts_mp3', 'zip', 'tts_mp3')
from google.colab import files
files.download('tts_mp3.zip')

Bỏ `tts_mp3.zip` vào thư mục **xai/video** rồi nhắn Claude để gắn vào video.